In [1]:
!git clone https://github.com/kasparvonbeelen/contracts.git


Cloning into 'contracts'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 118 (delta 56), reused 86 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 6.53 MiB | 23.98 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/contracts
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confli

Processing ./.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
^C


In [3]:
%cd contracts

/content/contracts


In [4]:
!pip install -q -e .

Obtaining file:///content/contracts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of tous-contract-analysis to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following yanked versions: 1.11.0, 1.14.0rc1
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python >=3.8,<3.12; 1.10.0rc1 Requires-Python >=3.8,<3.12; 1.10.0rc2 Requires-Python >=3.8,<3.12; 1.10.1 Requires-Python >=3.8,<3.12; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 1.7.2 Requires-Python >=3.7,<3.11; 1.7.3 Requires-Python >=3.7,<3.11; 1.8.0 Requires-Python >=3.8,<3.11; 1.8.0rc1 Requires-Python >=3.8,<3.11; 1.8.0rc2 Requires-Pyt

In [1]:
from tools.influence_analysis_helpers import visualize_sentence_similarity_timeline
# Create interactive output
from ipywidgets import interactive, IntSlider, FloatSlider
import pandas as pd
import numpy as np

ModuleNotFoundError: No module named 'tools'

In [ ]:
clause_type = "modification"  # privacy, liability, termination, indemnity, warranty
data_path = "results/tous"
influence_scores = pd.read_csv(f"{data_path}/influence_scores_{clause_type}.tsv" , sep="\t")
nodes_df = pd.read_csv(f"{data_path}/nodes_{clause_type}.tsv" , sep="\t")
emb_norm = np.loadtxt(f'{data_path}/emb_norm_{clause_type}.txt', delimiter=',')

In [ ]:


# Get top influential sentence IDs
top_ids = influence_scores.nlargest(1000, 'influence_score')['node_id'].values

sentence_idx_s = IntSlider(value=int(top_ids[0]), min=0, max=len(nodes_df)-1, step=1, description='Sentence:', layout={'width': '700px'})
min_sim_val = FloatSlider(value=0.80, min=0.50, max=0.99, step=0.01, description='Min Sim:', layout={'width': '400px'})

def show_timeline(sent_idx, min_sim):
    row = nodes_df.iloc[sent_idx]
    infl_score = influence_scores.iloc[sent_idx]['influence_score']

    print(f"\n{'='*100}")
    print(f"ROOT SENTENCE: {row['platform'].upper()} ({int(row['year'])}) | Influence Score: {infl_score:.2f}")
    print(f"{'='*100}")
    print(f"{row['sentence'][:400]}{'...' if len(row['sentence']) > 400 else ''}\n")

    fig, res = visualize_sentence_similarity_timeline(sent_idx, nodes_df, emb_norm, min_similarity=min_sim)
    if fig:
        print(f"Found {len(res)-1} similar sentences with similarity >= {min_sim:.2f}\n")
        fig.show()
    else:
        print(f"No similar sentences found with similarity >= {min_sim:.2f}\n")

interactive_plot = interactive(show_timeline, sent_idx=sentence_idx_s, min_sim=min_sim_val)
display(interactive_plot)

interactive(children=(IntSlider(value=3356, description='Sentence:', layout=Layout(width='700px'), max=6248), …